# 🔍 DeepSight — Training on Kaggle (T4 / P100 GPU)

Trains the DeepSight dual-branch image-forgery detection model on Kaggle.

### Before you run
1. **Settings (right panel) → Accelerator → GPU T4 x2** (or P100). One GPU is used.
2. **Settings → Internet → ON** (needed for `pip install` + downloading pretrained EfficientNet weights).
3. **Add the CASIA v2 dataset** via *Add Input* (e.g. the `casia-20-image-tampering-detection-dataset` dataset). The cell below auto-detects any input folder that contains `Au*/` and `Tp*/`.
4. **Add the project code** one of two ways:
   - *Git clone* (public repo): set `REPO_URL` in Cell 3, or
   - *Add Input*: upload the `deepsight` repo as a Kaggle Dataset, then Cell 3 finds it under `/kaggle/input/` and copies it into the writable working dir.

Outputs (checkpoints, logs) land in `/kaggle/working/deepsight/` and are saved automatically when you commit the notebook.

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────────────
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Settings → Accelerator → GPU T4 x2')

In [ ]:
# ── Cell 2: Locate / fetch the project, copy into writable working dir ──────────────
# Kaggle's /kaggle/input is READ-ONLY. Training writes to data/splits, models/, logs/,
# so the project must live under /kaggle/working (writable).
import os, shutil, subprocess
from pathlib import Path

WORK_DIR = Path('/kaggle/working/deepsight')

# Option A: clone from your public GitHub repo (default).
REPO_URL = 'https://github.com/hamzasterical/deepsight.git'
REPO_BRANCH = 'working'   # 'working' has the latest accuracy commits; 'main' is older

def looks_like_project(p: Path) -> bool:
    return (p / 'train.py').exists() and (p / 'src').is_dir() and (p / 'configs').is_dir()

if WORK_DIR.exists() and looks_like_project(WORK_DIR):
    print(f'Project already present at {WORK_DIR}')
elif REPO_URL:
    print(f'Cloning {REPO_URL} ({REPO_BRANCH}) ...')
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, str(WORK_DIR)], check=True)
else:
    # Option B: find the project among the added Kaggle datasets in /kaggle/input
    src = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if looks_like_project(Path(root)):
            src = Path(root)
            break
    if src is None:
        raise FileNotFoundError(
            'Project not found. Either set REPO_URL above, or add the deepsight '
            'repo as a Kaggle Dataset via "Add Input".')
    print(f'Found project at {src} → copying to {WORK_DIR}')
    shutil.copytree(src, WORK_DIR, dirs_exist_ok=True)

os.chdir(WORK_DIR)
print('Working directory:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

In [ ]:
# ── Cell 3: Install the extra dependencies ────────────────────────────────────
# Kaggle ships torch/torchvision with CUDA — only install the project's extras.
# albumentations >= 2.0 is required (the augmentation code uses the 2.x API).
!pip install -q "timm>=0.9.0" "albumentations>=2.0.0" "pillow-heif>=0.13.0" "scikit-image>=0.22.0"
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Auto-detect the CASIA v2 dataset and link it to data/raw/CASIA_v2 ────────
# The project scans data/raw/CASIA_v2/ for Au*/ (authentic) and Tp*/ (tampered).
import os
from pathlib import Path

AU_NAMES = {'au', 'au_jpg'}
TP_NAMES = {'tp', 'tp_jpg'}

def find_casia_root(search_root='/kaggle/input'):
    for root, dirs, _ in os.walk(search_root):
        lower = {d.lower(): d for d in dirs}
        has_au = any(n in lower for n in AU_NAMES)
        has_tp = any(n in lower for n in TP_NAMES)
        if has_au and has_tp:
            return Path(root)
    return None

casia_root = find_casia_root()
if casia_root is None:
    raise FileNotFoundError(
        'CASIA v2 not found under /kaggle/input. Add a CASIA v2 dataset via "Add Input". '
        'It must contain Au/ (or Au_jpg/) and Tp/ (or Tp_jpg/) folders.')

os.makedirs('data/raw', exist_ok=True)
link = Path('data/raw/CASIA_v2')
if link.exists() or link.is_symlink():
    link.unlink()
os.symlink(casia_root, link)
print(f'Linked CASIA root: {casia_root} → {link}')

# Report counts
for sub in sorted(os.listdir(casia_root)):
    p = casia_root / sub
    if p.is_dir() and sub.lower() in (AU_NAMES | TP_NAMES):
        n = sum(1 for _ in p.rglob('*') if _.is_file())
        print(f'  {sub:10s}: {n} files')

In [ ]:
# ── Cell 5: Quick sanity check — one dummy batch through the model ─────────────────
import sys, yaml, torch
from pathlib import Path
if str(Path('.').resolve()) not in sys.path:
    sys.path.insert(0, str(Path('.').resolve()))

config = yaml.safe_load(Path('configs/config.yaml').read_text())
print('Config:')
for k in ('batch_size', 'num_workers', 'phase1_epochs', 'phase2_epochs'):
    print(f'  {k:14s}: {config["training"][k]}')
print(f'  backbone      : {config["model"]["backbone"]}')

from src.models.dual_branch import DualBranchModel
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DualBranchModel(
    backbone=config['model']['backbone'],
    pretrained_rgb=True, pretrained_noise=False,
    feature_dim=config['model']['feature_dim'],
    hidden_dim=config['model']['hidden_dim'],
    dropout=config['model']['dropout'],
    freeze_bn=config['training']['freeze_bn'],
).to(device)
out = model(torch.randn(2, 3, 224, 224).to(device), torch.randn(2, 33, 224, 224).to(device))
print(f'\nForward OK. Output shape: {tuple(out.shape)}  (expected (2, 1))')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')
del model; torch.cuda.empty_cache()

In [ ]:
# ── Cell 6: Test the dataloader on one real batch ──────────────────────────────
import yaml
from pathlib import Path
from src.preprocessing.dataset_builder import create_dataloaders
from src.preprocessing.srm_filters import SRMFilterLayer

config = yaml.safe_load(Path('configs/config.yaml').read_text())
srm_layer = SRMFilterLayer().to(device)
train_loader, val_loader, test_loader = create_dataloaders(
    config, batch_size=4, num_workers=0, srm_layer=srm_layer)

print(f'Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}')
batch = next(iter(train_loader))
print(f"  rgb   : {tuple(batch['rgb'].shape)}")
print(f"  noise : {tuple(batch['noise'].shape)}")
print(f"  label : {batch['label'].flatten().tolist()}")
print('Dataloader OK ✓')
del train_loader, val_loader, test_loader, srm_layer; torch.cuda.empty_cache()

In [ ]:
# ── Cell 7: START TRAINING ─────────────────────────────────────────────────
# Phase 1 (10 epochs): warms up the noise branch + heads (RGB frozen)
# Phase 2 (40 epochs): full dual-branch fine-tuning
# Checkpoints → models/checkpoints/  (best at models/checkpoints/best_model.pth)
#
# Kaggle GPU sessions have a 9h (interactive) / 12h (commit) wall-clock limit. If you
# hit it, re-run with Cell 8 to resume from the last checkpoint.
!python train.py

In [ ]:
# ── Cell 8 (Optional): Resume training from the best checkpoint ───────────────────
# !python train.py --resume models/checkpoints/best_model.pth

In [ ]:
# ── Cell 9 (Optional): Plot training curves ──────────────────────────────────
import pandas as pd, matplotlib.pyplot as plt
from pathlib import Path

log_path = Path('logs/training_log.csv')
if not log_path.exists():
    print('No training log yet. Run Cell 7 first.')
else:
    df = pd.read_csv(log_path)
    print(df.tail(10).to_string(index=False))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('DeepSight Training Curves', fontsize=14, fontweight='bold')
    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss')
    axes[0].plot(df['epoch'], df['val_loss'],   label='Val Loss')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(df['epoch'], df['val_auc'], color='green', label='Val AUC')
    axes[1].set_title('Validation AUC-ROC'); axes[1].legend(); axes[1].grid(True)
    axes[2].plot(df['epoch'], df['val_f1'],  color='orange', label='Val F1')
    axes[2].plot(df['epoch'], df['val_acc'], color='purple', label='Val Acc')
    axes[2].set_title('Val F1 & Accuracy'); axes[2].legend(); axes[2].grid(True)
    plt.tight_layout(); plt.savefig('logs/training_curves.png', dpi=150); plt.show()
    print('Saved → logs/training_curves.png')

In [ ]:
# ── Cell 10 (Optional): Surface the best checkpoint as a notebook output ────────────
# Everything under /kaggle/working is saved when you 'Save Version' (commit).
# Copy the best model to the working root so it's easy to find / download.
import shutil, os
best = 'models/checkpoints/best_model.pth'
if os.path.exists(best):
    shutil.copy(best, '/kaggle/working/best_model.pth')
    print('Best model copied → /kaggle/working/best_model.pth')
else:
    print(f'No checkpoint at {best} yet.')